# Introduction to Biophysics EM Toys

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HyeonjeYang/electripyyy_lab/blob/main/biophysics_em_toys.ipynb)

A compact simulation notebook for Seoul National University's Summer 2026 undergraduate Introduction to Biophysics course.


In [ ]:
!pip install -q numpy scipy matplotlib ipywidgets

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from scipy import sparse
from scipy.sparse import linalg as spla
from scipy.special import j1
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, Button, Output

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})
rng = np.random.default_rng(7)

eps0 = 8.8541878128e-12
mu0 = 4 * np.pi * 1e-7
c = 1 / np.sqrt(eps0 * mu0)
F = 96485.33212
R = 8.314462618
T = 298.15
e = 1.602176634e-19
NA = 6.02214076e23
kB = R / NA


def note(text):
    display(HTML(f"<p><b>What to observe:</b> {text}</p>"))

def laplacian_1d(n, dx):
    main = -2 * np.ones(n)
    off = np.ones(n - 1)
    return sparse.diags([off, main, off], [-1, 0, 1], format="csr") / dx**2


def laplacian_2d(nx, ny, dx, dy=None):
    dy = dx if dy is None else dy
    Lx = laplacian_1d(nx, dx)
    Ly = laplacian_1d(ny, dy)
    return sparse.kron(sparse.eye(ny), Lx) + sparse.kron(Ly, sparse.eye(nx))


def solve_poisson_1d(rho, dx, eps=eps0, phi_left=0.0, phi_right=0.0):
    rho = np.asarray(rho, dtype=float)
    b = -rho / eps
    b[0] -= phi_left / dx**2
    b[-1] -= phi_right / dx**2
    return spla.spsolve(laplacian_1d(rho.size, dx), b)


def apply_dirichlet_bc(phi, bc):
    if not bc:
        return phi
    if "left" in bc:
        phi[:, 0] = bc["left"]
    if "right" in bc:
        phi[:, -1] = bc["right"]
    if "bottom" in bc:
        phi[0, :] = bc["bottom"]
    if "top" in bc:
        phi[-1, :] = bc["top"]
    return phi


def sor_poisson_2d(rho, dx, dy=None, eps=eps0, bc=None, omega=1.7, tol=1e-5, max_iter=6000):
    dy = dx if dy is None else dy
    rho = np.asarray(rho, dtype=float)
    phi = np.zeros_like(rho)
    apply_dirichlet_bc(phi, bc)
    idx2, idy2 = 1 / dx**2, 1 / dy**2
    denom = 2 * (idx2 + idy2)
    ny, nx = rho.shape
    jj, ii = np.indices((ny - 2, nx - 2))
    masks = ((ii + jj) % 2 == 0, (ii + jj) % 2 == 1)
    for it in range(max_iter):
        max_update = 0.0
        for mask in masks:
            proposal = ((phi[1:-1, 2:] + phi[1:-1, :-2]) * idx2
                        + (phi[2:, 1:-1] + phi[:-2, 1:-1]) * idy2
                        + rho[1:-1, 1:-1] / eps) / denom
            interior = phi[1:-1, 1:-1]
            updated = (1 - omega) * interior + omega * proposal
            max_update = max(max_update, np.max(np.abs(updated[mask] - interior[mask])))
            interior[mask] = updated[mask]
        apply_dirichlet_bc(phi, bc)
        if max_update < tol:
            break
    return phi, it + 1

def electric_field(phi, dx, dy=None):
    dy = dx if dy is None else dy
    dphi_dy, dphi_dx = np.gradient(phi, dy, dx)
    return -dphi_dx, -dphi_dy


def coulomb_phi_grid(X, Y, charges, eps=eps0, softening=1e-9):
    phi = np.zeros_like(X, dtype=float)
    for q, x0, y0 in charges:
        r = np.sqrt((X - x0) ** 2 + (Y - y0) ** 2 + softening**2)
        phi += q / (4 * np.pi * eps * r)
    return phi


def gaussian(x, center, width, area=1.0):
    y = np.exp(-0.5 * ((x - center) / width) ** 2)
    return area * y / np.trapz(y, x)


def crank_nicolson_step(u, D, v, dx, dt):
    '''One closed-boundary advection-diffusion step: dc/dt = D c_xx - v c_x.'''
    u = np.asarray(u, dtype=float)
    n = u.size
    main = -2 * D / dx**2 * np.ones(n)
    lower = (D / dx**2 + v / (2 * dx)) * np.ones(n - 1)
    upper = (D / dx**2 - v / (2 * dx)) * np.ones(n - 1)
    L = sparse.diags([lower, main, upper], [-1, 0, 1], format="csr")
    I = sparse.eye(n, format="csr")
    return spla.spsolve(I - 0.5 * dt * L, (I + 0.5 * dt * L) @ u)


def fdtd_1d_step(E, H, eps_r, dx, dt):
    H += dt / (mu0 * dx) * (E[1:] - E[:-1])
    E[1:-1] += dt / (eps0 * eps_r[1:-1] * dx) * (H[1:] - H[:-1])
    E[0], E[-1] = E[1], E[-2]
    return E, H


def line_animation(x, frames, ylabel, title, ylim=None, interval=45, xlabel="position"):
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    line, = ax.plot(x, frames[0], lw=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    else:
        y = np.asarray(frames)
        ax.set_ylim(1.05 * np.nanmin(y), 1.05 * np.nanmax(y))

    def update(i):
        line.set_ydata(frames[i])
        ax.set_title(f"{title}  frame={i:02d}")
        return (line,)

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())



def trajectory_animation(x, y, xlabel, ylabel, title, interval=35):
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.plot(x, y, color="0.75", lw=1)
    trace, = ax.plot([], [], color="tab:blue", lw=2)
    point, = ax.plot([], [], "o", color="tab:red")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.axis("equal")
    pad_x = 0.06 * max(np.ptp(x), 1e-12)
    pad_y = 0.06 * max(np.ptp(y), 1e-12)
    ax.set_xlim(np.min(x) - pad_x, np.max(x) + pad_x)
    ax.set_ylim(np.min(y) - pad_y, np.max(y) + pad_y)

    def update(i):
        trace.set_data(x[:i+1], y[:i+1])
        point.set_data([x[i]], [y[i]])
        ax.set_title(f"{title}  frame={i:02d}")
        return trace, point

    anim = animation.FuncAnimation(fig, update, frames=len(x), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def debye_length(I_mM, eps_r=78.5):
    I_molm3 = np.asarray(I_mM, dtype=float)
    return np.sqrt(eps_r * eps0 * R * T / (2 * F**2 * I_molm3))


def fit_exp_decay_length(x, y):
    y = np.asarray(y)
    mask = y > np.max(y) * 1e-3
    slope, intercept = np.polyfit(x[mask], np.log(y[mask]), 1)
    return -1 / slope


## 1. Fields & potential (slides 2-4)

Point charges define `phi`; the field is `E = -grad(phi)`.

$$
\phi(\mathbf r)=\sum_i {q_i \over 4\pi\epsilon |\mathbf r-\mathbf r_i|},
\qquad
\mathbf E=-\nabla\phi .
$$


In [ ]:
x = np.linspace(-40e-9, 40e-9, 81)
y = np.linspace(-40e-9, 40e-9, 81)
X, Y = np.meshgrid(x, y)
test_phi = coulomb_phi_grid(np.array([[20e-9]]), np.array([[0.0]]), [(e, 0.0, 0.0)], softening=0.0)
assert np.isclose(test_phi[0, 0], e / (4 * np.pi * eps0 * 20e-9), rtol=1e-12)


def plot_fields(separation_nm=18.0, q_e=1.0):
    d = separation_nm * 1e-9
    charges = [(q_e * e, -d / 2, 0.0), (-q_e * e, d / 2, 0.0)]
    phi = coulomb_phi_grid(X, Y, charges, softening=2e-9)
    Ex, Ey = electric_field(phi, x[1] - x[0])
    phi_mV = 1e3 * phi
    stride = 5
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(np.clip(phi_mV, -150, 150), extent=[x.min()*1e9, x.max()*1e9, y.min()*1e9, y.max()*1e9],
                   origin="lower", cmap="coolwarm", aspect="equal")
    fig.colorbar(im, ax=ax, label="phi (mV)")
    levels = np.linspace(-120, 120, 13)
    ax.contour(X * 1e9, Y * 1e9, np.clip(phi_mV, -150, 150), levels=levels, colors="k", linewidths=0.6, alpha=0.55)
    scale = np.percentile(np.hypot(Ex, Ey), 98)
    ax.quiver(X[::stride, ::stride] * 1e9, Y[::stride, ::stride] * 1e9,
              Ex[::stride, ::stride] / scale, Ey[::stride, ::stride] / scale,
              color="white", pivot="mid", scale=22)
    ax.scatter([-d/2*1e9, d/2*1e9], [0, 0], c=["tab:red", "tab:blue"], s=60, edgecolor="k")
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("y (nm)")
    ax.set_title("Dipole potential and numerical E field")
    plt.show()


interact(plot_fields,
         separation_nm=FloatSlider(value=18, min=4, max=36, step=2, description="d (nm)"),
         q_e=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.2, description="q/e"))
note('Dipole separation changes equipotentials; E stays perpendicular to them.')


## 2. Poisson & Laplace (slides 5-7)

Laplace has no source term; Poisson adds charge density.

$$
\nabla^2\phi=0,
\qquad
\nabla^2\phi=-{\rho\over\epsilon}.
$$


In [ ]:
L = 10e-6
n = 96
dx1 = L / (n + 1)
xi = np.linspace(dx1, L - dx1, n)
rho0 = 1.5e-4
eps_water = 80 * eps0
phi_num = solve_poisson_1d(rho0 * np.ones(n), dx1, eps=eps_water)
phi_exact = rho0 * xi * (L - xi) / (2 * eps_water)
assert np.max(np.abs(phi_num - phi_exact)) < 1e-12


def plot_poisson_laplace(V_mV=80.0, patch_rho_mC_m3=2.0):
    nx, ny = 72, 54
    width = 120e-6
    dx = width / (nx - 1)
    xs = np.linspace(0, width, nx)
    ys = np.linspace(0, dx * (ny - 1), ny)
    boundary = np.linspace(V_mV * 1e-3 / 2, -V_mV * 1e-3 / 2, nx)
    bc = {"left": V_mV * 1e-3 / 2, "right": -V_mV * 1e-3 / 2, "bottom": boundary, "top": boundary}
    rho_lap = np.zeros((ny, nx))
    rho_patch = np.zeros((ny, nx))
    yy, xx = np.meshgrid(ys, xs, indexing="ij")
    patch = ((xx - 0.55 * width) ** 2 + (yy - 0.55 * ys.max()) ** 2) < (0.13 * width) ** 2
    rho_patch[patch] = patch_rho_mC_m3 * 1e-3
    phi_lap, it_lap = sor_poisson_2d(rho_lap, dx, eps=eps_water, bc=bc, tol=2e-7)
    phi_poi, it_poi = sor_poisson_2d(rho_patch, dx, eps=eps_water, bc=bc, tol=2e-7)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    for ax, phi, title, its in [(axes[0], phi_lap, "Laplace: rho=0", it_lap), (axes[1], phi_poi, "Poisson: charged patch", it_poi)]:
        im = ax.imshow(1e3 * phi, extent=[0, width*1e6, 0, ys.max()*1e6], origin="lower", cmap="coolwarm", aspect="auto")
        ax.contour(xs * 1e6, ys * 1e6, 1e3 * phi, colors="k", linewidths=0.55, alpha=0.6)
        ax.set_xlabel("x (um)")
        ax.set_ylabel("y (um)")
        ax.set_title(f"{title}\nSOR iterations={its}")
        fig.colorbar(im, ax=ax, label="phi (mV)")
    plt.show()


fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(xi * 1e6, 1e3 * phi_num, label="finite difference")
ax.plot(xi * 1e6, 1e3 * phi_exact, "--", label="analytic")
ax.set_xlabel("x (um)")
ax.set_ylabel("phi (mV)")
ax.set_title("1D Poisson check")
ax.legend()
plt.show()

interact(plot_poisson_laplace,
         V_mV=FloatSlider(value=80, min=10, max=160, step=10, description="V (mV)"),
         patch_rho_mC_m3=FloatSlider(value=2.0, min=-4.0, max=4.0, step=0.5, description="rho"))
note('Charge bends equipotentials away from the Laplace solution.')


## 3. Gel electrophoresis (slide 6)

A linear gel potential gives a uniform field and constant band velocity.

$$
{d^2\phi\over dx^2}=0,\qquad \phi(0)=V_0,\quad\phi(L)=0,\qquad v=\mu_{\rm ep}E .
$$


In [ ]:
xg = np.linspace(0, 0.05, 300)
V0_check = 100.0
phi_check = V0_check * (1 - xg / xg.max())
E_check = -np.gradient(phi_check, xg)
assert np.allclose(E_check[2:-2], V0_check / xg.max(), rtol=1e-3)


def plot_gel(V0=100.0, L_cm=5.0, mu_ep_1e8=3.0):
    L = L_cm * 1e-2
    x = np.linspace(0, L, 320)
    phi = V0 * (1 - x / L)
    E = V0 / L
    mu_ep = mu_ep_1e8 * 1e-8
    v = mu_ep * E
    width = 0.04 * L
    x0 = 0.16 * L
    t_end = 0.72 * (L - x0) / max(v, 1e-12)
    times = np.linspace(0, t_end, 45)
    frames = np.array([gaussian(x, x0 + v * t, width) for t in times])
    fig, ax1 = plt.subplots(figsize=(6.2, 3.2))
    ax1.plot(x * 100, phi, color="tab:blue")
    ax1.set_xlabel("x (cm)")
    ax1.set_ylabel("phi (V)", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(x * 100, -np.gradient(phi, x), color="tab:red")
    ax2.set_ylabel("E (V/m)", color="tab:red")
    ax1.set_title(f"Linear gel potential; band speed = {v*1e6:.2f} um/s")
    plt.show()
    display(line_animation(x * 100, frames, "band concentration (a.u.)", "Electrophoretic drift", interval=55, xlabel="x (cm)"))


interact(plot_gel,
         V0=FloatSlider(value=100, min=20, max=200, step=10, description="V0 (V)"),
         L_cm=FloatSlider(value=5, min=2, max=12, step=1, description="L (cm)"),
         mu_ep_1e8=FloatSlider(value=3, min=0.5, max=8, step=0.5, description="mu_ep"))
note('Larger E moves the band faster.')


## 4. Charge relaxation (slide 8)

Conductive media erase free charge on the time scale `tau = eps/sigma`.

$$
{d\rho\over dt}=-{\sigma\over\epsilon}\rho,
\qquad
\rho(t)=\rho_0 e^{-t/\tau},\quad \tau={\epsilon\over\sigma}.
$$


In [ ]:
tau_saline = 80 * eps0 / 1.0
assert np.isclose(tau_saline * 1e9, 0.708, rtol=0.02)


def plot_relaxation(sigma=1.0, eps_r=80.0):
    eps = eps_r * eps0
    tau = eps / sigma
    t = np.linspace(0, 6 * tau, 220)
    rho = np.exp(-t / tau)
    assert np.isclose(rho[np.argmin(np.abs(t - tau))], np.exp(-1), rtol=0.05)
    fig, ax = plt.subplots(figsize=(5.8, 3.2))
    ax.plot(t * 1e9, rho, lw=2)
    ax.axvline(tau * 1e9, color="k", ls="--", label=f"tau = {tau*1e9:.3f} ns")
    ax.set_xlabel("time (ns)")
    ax.set_ylabel("rho / rho0")
    ax.set_title("Charge relaxation")
    ax.legend()
    plt.show()
    x = np.linspace(-25, 25, 220)
    base = np.exp(-0.5 * (x / 5) ** 2)
    frames = np.array([amp * base for amp in np.linspace(1, np.exp(-5), 40)])
    display(line_animation(x, frames, "charge density (a.u.)", "Relaxing charge blob", ylim=(0, 1.05), interval=55, xlabel="x (um)"))


interact(plot_relaxation,
         sigma=FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="sigma"),
         eps_r=FloatSlider(value=80.0, min=2.0, max=100.0, step=2.0, description="eps_r"))
note('Larger sigma makes charge vanish faster.')


## 5. Why EQS is reasonable (slide 5)

Compare relaxation, diffusion, and transit times before using EQS.

$$
\tau_e={\epsilon\over\sigma},\qquad
\tau_m=\mu\sigma L^2,\qquad
\tau_{\rm em}={L\over c}.
$$


In [ ]:
def plot_eqs(t_obs_us=1.0, sigma=1.0, eps_r=80.0):
    eps = eps_r * eps0
    Ls = np.logspace(-7, 0, 260)
    tau_e = eps / sigma * np.ones_like(Ls)
    tau_m = mu0 * sigma * Ls**2
    tau_em = Ls / c
    t_obs = t_obs_us * 1e-6
    eqs_ok = t_obs > 10 * np.maximum(tau_e, tau_em)
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ax.loglog(Ls, tau_e, label="charge relaxation tau_e")
    ax.loglog(Ls, tau_m, label="magnetic diffusion tau_m")
    ax.loglog(Ls, tau_em, label="EM transit tau_em")
    ax.axhline(t_obs, color="k", ls="--", label="observation time")
    ax.fill_between(Ls, 1e-18, 1e4, where=eqs_ok, color="tab:green", alpha=0.12, label="EQS comfortable")
    for label, L0 in [("cell 10 um", 10e-6), ("tissue 1 cm", 1e-2)]:
        ax.scatter([L0], [L0 / c], color="tab:red")
        ax.text(L0 * 1.1, L0 / c * 1.6, label, fontsize=8)
    ax.set_xlabel("length L (m)")
    ax.set_ylabel("time scale (s)")
    ax.set_ylim(1e-18, 1e4)
    ax.set_title("EQS time-scale comparison")
    ax.legend(fontsize=8)
    plt.show()
    for label, L0 in [("cell", 10e-6), ("tissue", 1e-2)]:
        print(f"{label}: tau_em={L0/c:.2e} s, tau_e={eps/sigma:.2e} s, tau_m={mu0*sigma*L0**2:.2e} s")


interact(plot_eqs,
         t_obs_us=FloatSlider(value=1.0, min=0.001, max=1000.0, step=0.5, description="t_obs us"),
         sigma=FloatSlider(value=1.0, min=0.01, max=2.0, step=0.05, description="sigma"),
         eps_r=FloatSlider(value=80.0, min=2.0, max=100.0, step=2.0, description="eps_r"))
note('EQS is better when observation time is much longer than tau_e and tau_em.')


## 6. Nernst-Planck migration (slides 10-11)

Ion flux is diffusion plus field-driven migration.

$$
J=-D{dc\over dx}-{zFD\over RT}c{d\phi\over dx},
\qquad
{|\hbox{migration}|\over|\hbox{diffusion}|}\sim {|z|F|\Delta\phi|\over RT}.
$$


In [ ]:
thermal_mV = R * T / F * 1e3
assert np.isclose(thermal_mV, 25.69, rtol=0.01)


def plot_nernst_planck(delta_mV=25.7, z=1, D_um2_s=100.0):
    L = 100e-6
    x = np.linspace(0, L, 360)
    D = D_um2_s * 1e-12
    delta_phi = delta_mV * 1e-3
    Efield = delta_phi / L
    mobility = z * F * D / (R * T)
    v = mobility * Efield
    width0 = 7e-6
    x0 = 0.35 * L
    t_end = min(0.45 * L / max(abs(v), 1e-10), 4.0)
    times = np.linspace(0, t_end, 5)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
    for t_s in times:
        width = np.sqrt(width0**2 + 2 * D * t_s)
        center = np.clip(x0 + v * t_s, 0.08 * L, 0.92 * L)
        axes[0].plot(x * 1e6, gaussian(x, center, width), label=f"{t_s:.2f} s")
    axes[0].set_xlabel("x (um)")
    axes[0].set_ylabel("concentration (a.u.)")
    axes[0].set_title(f"Drift-diffusion packet, v={v*1e6:.2f} um/s")
    axes[0].legend(fontsize=8)
    V = np.linspace(0, 90, 200)
    ratio = abs(z) * V / thermal_mV
    axes[1].plot(V, ratio)
    axes[1].axhline(1, color="k", ls="--")
    axes[1].axvline(thermal_mV / max(abs(z), 1), color="tab:red", ls=":", label="RT/(|z|F)")
    axes[1].scatter([abs(delta_mV)], [abs(z) * abs(delta_mV) / thermal_mV], color="tab:red")
    axes[1].set_xlabel("|Delta phi| (mV)")
    axes[1].set_ylabel("migration / diffusion scale")
    axes[1].set_title("Voltage scale for migration")
    axes[1].legend(fontsize=8)
    plt.show()


interact(plot_nernst_planck,
         delta_mV=FloatSlider(value=25.7, min=-90, max=90, step=5, description="Delta mV"),
         z=IntSlider(value=1, min=-3, max=3, step=1, description="z"),
         D_um2_s=FloatSlider(value=100, min=10, max=500, step=10, description="D"))
note('Migration becomes comparable to diffusion near RT/F.')


## 7. Poisson-Boltzmann & Debye length (slides 12-13)

Electrolytes screen surface potential over the Debye length.

$$
{d^2\phi\over dx^2}=\kappa^2\phi,\qquad
\phi(x)=\phi_0 e^{-x/\lambda_D},\qquad
\lambda_D={1\over\kappa}.
$$


In [ ]:
assert np.isclose(debye_length(100) * 1e9, 0.96, rtol=0.08)


def solve_nonlinear_pb(phi0, I_mM, eps_r=78.5):
    eps = eps_r * eps0
    c0 = I_mM
    lam_D = debye_length(I_mM, eps_r)
    Lbox = 10 * lam_D
    n = 140
    dx = Lbox / (n + 1)
    x = np.linspace(dx, Lbox - dx, n)
    Lmat = laplacian_1d(n, dx).tolil()
    b = np.zeros(n)
    b[0] -= phi0 / dx**2
    phi = phi0 * np.exp(-x / lam_D)
    factor = 2 * F * c0 / eps
    alpha = F / (R * T)
    for _ in range(12):
        res = Lmat @ phi + b - factor * np.sinh(alpha * phi)
        J = Lmat - sparse.diags(factor * alpha * np.cosh(alpha * phi), format="lil")
        step = spla.spsolve(J.tocsr(), -res)
        phi += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return x, phi


def plot_pb(I_mM=10.0, phi0_mV=25.0, show_nonlinear=True):
    lam_D = debye_length(I_mM)
    x = np.linspace(0, 10 * lam_D, 240)
    phi0 = phi0_mV * 1e-3
    phi_lin = phi0 * np.exp(-x / lam_D)
    lam_fit = fit_exp_decay_length(x, np.abs(phi_lin))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
    axes[0].plot(x * 1e9, phi_lin * 1e3, label=f"linear, fit lam={lam_fit*1e9:.2f} nm")
    if show_nonlinear:
        xn, phin = solve_nonlinear_pb(phi0, I_mM)
        axes[0].plot(xn * 1e9, phin * 1e3, "--", label="nonlinear Newton")
    axes[0].set_xlabel("x (nm)")
    axes[0].set_ylabel("phi (mV)")
    axes[0].set_title(f"Debye screening: lam_D={lam_D*1e9:.2f} nm")
    axes[0].legend(fontsize=8)
    I = np.logspace(0, 3, 160)
    axes[1].loglog(I, debye_length(I) * 1e9)
    axes[1].scatter([I_mM], [lam_D * 1e9], color="tab:red")
    axes[1].set_xlabel("ionic strength I (mM)")
    axes[1].set_ylabel("lam_D (nm)")
    axes[1].set_title("lam_D scales as I^{-1/2}")
    plt.show()


interact(plot_pb,
         I_mM=FloatSlider(value=10, min=1, max=300, step=5, description="I mM"),
         phi0_mV=FloatSlider(value=25, min=1, max=100, step=5, description="phi0"),
         show_nonlinear=Checkbox(value=True, description="nonlinear"))
note('Higher ionic strength shortens the screening length.')


## 8. Virus as a charged sphere (slides 14-16)

A charged sphere gives screened potential and drift-diffusion transport.

$$
\phi(r)={Q e^{-\kappa(r-a)}\over 4\pi\epsilon(1+\kappa a)r},
\qquad
{\partial c\over\partial t}=-{\partial\over\partial x}\left(vc-D{\partial c\over\partial x}\right).
$$


In [ ]:
def virus_phi(r, a, Q, kappa, eps=80 * eps0):
    return Q * np.exp(-kappa * (r - a)) / (4 * np.pi * eps * (1 + kappa * a) * r)


def virus_transport(absorbing=True, z_eff=-20, field_V_cm=10.0, D_um2_s=5.0):
    L = 120e-6
    n = 220
    x = np.linspace(0, L, n)
    dx = x[1] - x[0]
    D = D_um2_s * 1e-12
    Efield = field_V_cm * 100
    q_eff = z_eff * e
    v = q_eff * D * Efield / (kB * T)
    dt = min(0.35 * dx**2 / max(D, 1e-18), 0.35 * dx / max(abs(v), 1e-12))
    steps = int(np.ceil(8.0 / dt))
    dt = 8.0 / steps
    c0 = gaussian(x, 0.72 * L, 0.08 * L)
    profiles, masses, times = [], [], []
    cnow = c0.copy()
    capture = set(np.linspace(0, steps, 7, dtype=int))
    for step in range(steps + 1):
        if step in capture:
            profiles.append(cnow.copy())
            masses.append(np.trapz(cnow, x))
            times.append(step * dt)
        flux = np.zeros(n + 1)
        c_up = cnow[1:] if v < 0 else cnow[:-1]
        flux[1:-1] = v * c_up - D * (cnow[1:] - cnow[:-1]) / dx
        if absorbing:
            flux[0] = min(v * cnow[0], 0.0) - D * (cnow[0] - 0.0) / (0.5 * dx)
        else:
            flux[0] = 0.0
        flux[-1] = 0.0
        cnow = np.maximum(cnow - dt * (flux[1:] - flux[:-1]) / dx, 0)
    return x, np.array(profiles), np.array(masses), np.array(times), v


def plot_virus(a_nm=50.0, Q_e=-80.0, I_mM=10.0, field_V_cm=10.0):
    a = a_nm * 1e-9
    Q = Q_e * e
    lam_D = debye_length(I_mM)
    r = np.linspace(a, a + 12 * lam_D + 4 * a, 300)
    phi = virus_phi(r, a, Q, 1 / lam_D)
    assert np.isfinite(phi).all()
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)
    axes[0].plot((r - a) * 1e9, phi * 1e3)
    axes[0].set_xlabel("distance from surface (nm)")
    axes[0].set_ylabel("phi (mV)")
    axes[0].set_title(f"Spherical PB, lam_D={lam_D*1e9:.2f} nm")
    for absorbing, label, color in [(True, "absorbing membrane", "tab:red"), (False, "reflecting wall", "tab:blue")]:
        x, prof, mass, times, v = virus_transport(absorbing=absorbing, field_V_cm=field_V_cm)
        for p, t_s in zip(prof[::2], times[::2]):
            axes[1].plot(x * 1e6, p, color=color, alpha=0.35 + 0.08 * t_s, lw=1.4)
        axes[2].plot(times, mass / mass[0], color=color, label=label)
    axes[1].set_xlabel("x from membrane (um)")
    axes[1].set_ylabel("virus concentration (a.u.)")
    axes[1].set_title(f"Drift toward x=0, v={v*1e6:.2f} um/s")
    axes[2].set_xlabel("time (s)")
    axes[2].set_ylabel("remaining mass")
    axes[2].set_ylim(0, 1.05)
    axes[2].set_title("Boundary condition matters")
    axes[2].legend(fontsize=8)
    plt.show()


interact(plot_virus,
         a_nm=FloatSlider(value=50, min=15, max=120, step=5, description="a nm"),
         Q_e=FloatSlider(value=-80, min=-250, max=-5, step=5, description="Q/e"),
         I_mM=FloatSlider(value=10, min=1, max=200, step=5, description="I mM"),
         field_V_cm=FloatSlider(value=10, min=0, max=40, step=2, description="E V/cm"))
note('Absorbing boundaries remove virus mass; reflecting boundaries do not.')


## 9. Donnan equilibrium (slide 17)

Fixed charge shifts ion partitioning through a Donnan potential.

$$
c_+^{t}-c_-^{t}-X=0,\qquad
c_i^{t}=c_i^{b}\exp\left(-z_i {F\Delta\phi\over RT}\right).
$$


In [ ]:
def plot_donnan(fixed_mM=20.0, bath_mM=150.0, drug_z=1):
    X = fixed_mM
    cb = bath_mM
    psi = -np.arcsinh(X / (2 * cb))
    dphi = psi * R * T / F
    c_plus = cb * np.exp(-psi)
    c_minus = cb * np.exp(psi)
    assert abs(c_plus - c_minus - X) < 1e-10
    drug_bath = 1.0
    drug_tissue = drug_bath * np.exp(-drug_z * psi)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
    labels = ["cation", "anion", f"drug z={drug_z}"]
    bath = [cb, cb, drug_bath]
    tissue = [c_plus, c_minus, drug_tissue]
    xpos = np.arange(len(labels))
    axes[0].bar(xpos - 0.18, bath, width=0.36, label="bath")
    axes[0].bar(xpos + 0.18, tissue, width=0.36, label="tissue")
    axes[0].set_xticks(xpos, labels)
    axes[0].set_ylabel("concentration (mM; drug normalized)")
    axes[0].set_title(f"Donnan potential = {dphi*1e3:.2f} mV")
    axes[0].legend()
    zvals = np.arange(-3, 4)
    uptake = np.exp(-zvals * psi)
    axes[1].bar(zvals, uptake, color="tab:purple")
    axes[1].axhline(1, color="k", ls="--")
    axes[1].set_xlabel("drug charge z")
    axes[1].set_ylabel("tissue / bath")
    axes[1].set_title("Charge-selective uptake")
    plt.show()


interact(plot_donnan,
         fixed_mM=FloatSlider(value=20, min=0, max=80, step=5, description="fixed"),
         bath_mM=FloatSlider(value=150, min=20, max=300, step=10, description="bath"),
         drug_z=IntSlider(value=1, min=-3, max=3, step=1, description="drug z"))
note('Fixed negative charge enriches positive drugs.')


## 10. Lorentz, Hall, and crossed-field cycloids (slides 18-19)

Crossed `E` and `B` fields create cycloid drift; Hall voltage scales with `1/(nq)`.

$$
\mathbf F=q(\mathbf E+\mathbf v\times\mathbf B),
\qquad
E_H={J_x B_z\over nq}.
$$


In [ ]:
def boris_trajectory(B_mT=25.0, E_V_m=40.0, q_sign=1):
    q = q_sign * e
    m = 1.67262192369e-27
    B = np.array([0.0, 0.0, B_mT * 1e-3])
    Evec = np.array([0.0, E_V_m, 0.0])
    omega = abs(q) * np.linalg.norm(B) / m
    dt = 0.035 / max(omega, 1e3)
    steps = 1600
    r = np.zeros(3)
    v = np.array([700.0, 0.0, 0.0])
    out = np.zeros((steps, 3))
    for i in range(steps):
        v_minus = v + (q * Evec / m) * dt / 2
        tvec = (q * B / m) * dt / 2
        svec = 2 * tvec / (1 + np.dot(tvec, tvec))
        v_prime = v_minus + np.cross(v_minus, tvec)
        v_plus = v_minus + np.cross(v_prime, svec)
        v = v_plus + (q * Evec / m) * dt / 2
        r = r + v * dt
        out[i] = r
    return out


def crossed_field_cycloid(B_mT=25.0, E_V_m=40.0, cycles=3.0):
    B = B_mT * 1e-3
    Eabs = abs(E_V_m)
    m = 1.67262192369e-27
    omega = e * B / m
    v_d = Eabs / B
    t = np.linspace(0, cycles * 2 * np.pi / omega, 180)
    x = v_d * (t - np.sin(omega * t) / omega)
    y = np.sign(E_V_m) * v_d / omega * (1 - np.cos(omega * t))
    return t, x, y, v_d, omega


def plot_lorentz_hall(B_mT=25.0, E_V_m=40.0, Jx=1.0):
    traj = boris_trajectory(B_mT, E_V_m)
    t_c, x_c, y_c, v_d, omega = crossed_field_cycloid(B_mT, E_V_m)
    n = np.logspace(20, 25, 160)
    width = 1e-3
    VH = Jx * (B_mT * 1e-3) * width / (n * e)
    slope = np.polyfit(1 / (n * e), VH, 1)[0]
    assert np.isclose(slope, Jx * B_mT * 1e-3 * width, rtol=1e-12)
    assert np.isclose(x_c[-1], v_d * t_c[-1], rtol=0.02)
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.6), constrained_layout=True)
    axes[0].plot(traj[:, 0] * 1e3, traj[:, 1] * 1e3)
    axes[0].set_xlabel("x (mm)")
    axes[0].set_ylabel("y (mm)")
    axes[0].set_title("Boris Lorentz trajectory")
    axes[0].axis("equal")
    axes[1].plot(x_c * 1e3, y_c * 1e3, color="tab:green")
    axes[1].set_xlabel("x (mm)")
    axes[1].set_ylabel("y (mm)")
    axes[1].set_title(f"Crossed-field cycloid, v_d={v_d:.0f} m/s")
    axes[1].axis("equal")
    axes[2].plot(1 / (n * e), VH * 1e6)
    axes[2].set_xlabel("1 / (n q) (m^3/C)")
    axes[2].set_ylabel("V_H (uV)")
    axes[2].set_title("Single-carrier Hall scaling")
    plt.show()
    display(trajectory_animation(x_c * 1e3, y_c * 1e3, "x (mm)", "y (mm)", "Crossed E-B cycloid"))
    print("Electrolytes have positive and negative mobile carriers, so this single-carrier slope is not the whole story.")


interact(plot_lorentz_hall,
         B_mT=FloatSlider(value=25, min=2, max=80, step=2, description="B mT"),
         E_V_m=FloatSlider(value=40, min=-100, max=100, step=10, description="E V/m"),
         Jx=FloatSlider(value=1, min=0.1, max=5, step=0.1, description="Jx"))
note('B controls orbit size; E/B sets crossed-field drift.')


## 11. EM wave in 1D (slide 20)

Yee FDTD shows pulse reflection, transmission, and slower speed in dielectric media.

$$
{\partial E\over\partial t}={1\over\epsilon}{\partial H\over\partial x},\qquad
{\partial H\over\partial t}={1\over\mu}{\partial E\over\partial x},\qquad
n={c\over v}.
$$


In [ ]:
def run_fdtd(eps_slab=4.0, slab_width_cm=12.0):
    nx = 420
    length = 0.60
    x = np.linspace(0, length, nx)
    dx = x[1] - x[0]
    eps_r = np.ones(nx)
    center = 0.36
    half = slab_width_cm * 1e-2 / 2
    eps_r[(x > center - half) & (x < center + half)] = eps_slab
    dt = 0.95 * dx / (c * np.sqrt(eps_r.max()))
    E = np.zeros(nx)
    H = np.zeros(nx - 1)
    src = int(0.12 * nx)
    left_probe = int(0.22 * nx)
    right_probe = int(0.78 * nx)
    frames, left_trace, right_trace, times = [], [], [], []
    for step in range(760):
        pulse = np.exp(-0.5 * ((step - 65) / 16) ** 2)
        E[src] += pulse
        E, H = fdtd_1d_step(E, H, eps_r, dx, dt)
        left_trace.append(E[left_probe])
        right_trace.append(E[right_probe])
        times.append(step * dt)
        if step % 12 == 0:
            frames.append(E.copy())
    return x, np.array(frames), np.array(left_trace), np.array(right_trace), np.array(times), eps_r


def measure_index(eps_r_value):
    nx = 600
    length = 0.90
    x = np.linspace(0, length, nx)
    dx = x[1] - x[0]
    eps_r = eps_r_value * np.ones(nx)
    dt = 0.90 * dx / (c / np.sqrt(eps_r_value))
    E = np.exp(-0.5 * ((x - 0.16) / 0.028) ** 2)
    eta = np.sqrt(mu0 / (eps0 * eps_r_value))
    H = -0.5 * (E[1:] + E[:-1]) / eta
    positions, times = [], []
    for step in range(260):
        E, H = fdtd_1d_step(E, H, eps_r, dx, dt)
        if 30 <= step <= 210 and step % 10 == 0:
            positions.append(x[np.argmax(E)])
            times.append(step * dt)
    v_meas = np.polyfit(times, positions, 1)[0]
    return c / v_meas

def plot_fdtd(eps_slab=4.0, slab_width_cm=12.0):
    n_meas = measure_index(eps_slab)
    assert np.isclose(n_meas, np.sqrt(eps_slab), rtol=0.05)
    x, frames, left_trace, right_trace, times, eps_r = run_fdtd(eps_slab, slab_width_cm)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
    axes[0].plot(times * 1e9, left_trace, label="reflected-side probe")
    axes[0].plot(times * 1e9, right_trace, label="transmitted-side probe")
    axes[0].set_xlabel("time (ns)")
    axes[0].set_ylabel("E_z (a.u.)")
    axes[0].set_title(f"Measured n={n_meas:.2f}, expected {np.sqrt(eps_slab):.2f}")
    axes[0].legend(fontsize=8)
    axes[1].plot(x, eps_r)
    axes[1].set_xlabel("x (m)")
    axes[1].set_ylabel("relative permittivity")
    axes[1].set_title("Dielectric slab")
    plt.show()
    display(line_animation(x, frames, "E_z (a.u.)", "FDTD pulse and dielectric slab", ylim=(-1.2, 1.2), interval=45, xlabel="x (m)"))


interact(plot_fdtd,
         eps_slab=FloatSlider(value=4, min=1, max=9, step=0.5, description="eps_r"),
         slab_width_cm=FloatSlider(value=12, min=4, max=24, step=2, description="width cm"))
note('Higher eps_r slows the pulse and changes reflection/transmission.')


## 12. Wave optics & diffraction (slide 21)

Fraunhofer patterns come from aperture Fourier transforms; wavelength moves the fringes.

$$
I(\theta)\propto |\mathcal F\{A(x)\}|^2,\qquad
d\sin\theta=m\lambda,\qquad
\theta_R\simeq 1.22{\lambda\over D},\quad
d_{\min}={\lambda\over 2{\rm NA}} .
$$


In [ ]:
def aperture_1d(x, kind, slit_width, separation, count):
    A = np.zeros_like(x)
    if kind == "single slit":
        A[np.abs(x) < slit_width / 2] = 1
    elif kind == "double slit":
        for center in [-separation / 2, separation / 2]:
            A[np.abs(x - center) < slit_width / 2] = 1
    else:
        centers = (np.arange(count) - (count - 1) / 2) * separation
        for center in centers:
            A[np.abs(x - center) < slit_width / 2] = 1
    return A


def fraunhofer_pattern(kind, wavelength_nm, slit_width_um, separation_um, grating_lines=5):
    lam = wavelength_nm * 1e-9
    slit_width = slit_width_um * 1e-6
    separation = separation_um * 1e-6
    span = max(12 * separation, 80 * slit_width)
    n = 4096
    xa = np.linspace(-span / 2, span / 2, n)
    dx = xa[1] - xa[0]
    A = aperture_1d(xa, kind, slit_width, separation, grating_lines)
    fft = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(A)))
    sin_theta = np.fft.fftshift(np.fft.fftfreq(n, d=dx)) * lam
    valid = np.abs(sin_theta) < 0.35
    theta_deg = np.degrees(np.arcsin(sin_theta[valid]))
    I = np.abs(fft[valid]) ** 2
    return xa, A, theta_deg, I / I.max()


def double_slit_fraunhofer(theta_deg, wavelength_nm, slit_width_um, separation_um):
    theta = np.radians(theta_deg)
    lam = wavelength_nm * 1e-9
    a = slit_width_um * 1e-6
    d = separation_um * 1e-6
    beta_over_pi = a * np.sin(theta) / lam
    envelope = np.sinc(beta_over_pi) ** 2
    interference = np.cos(np.pi * d * np.sin(theta) / lam) ** 2
    I = envelope * interference
    return I / np.max(I)


assert np.isclose(double_slit_fraunhofer(np.array([0.0]), 550, 8, 24)[0], 1.0)


def plot_diffraction(kind="double slit", wavelength_nm=550.0, slit_width_um=8.0, separation_um=24.0, grating_lines=5, NA=1.0):
    lam = wavelength_nm * 1e-9
    slit_width = slit_width_um * 1e-6
    separation = separation_um * 1e-6
    xa, A, theta_deg, I = fraunhofer_pattern(kind, wavelength_nm, slit_width_um, separation_um, grating_lines)
    D = max(slit_width, 1e-12)
    airy_arg = np.linspace(1e-6, 18, 500)
    airy = (2 * j1(airy_arg) / airy_arg) ** 2
    theta_airy = airy_arg * lam / (np.pi * D)
    rayleigh = 1.22 * lam / D
    abbe = lam / (2 * NA)
    assert np.isclose(rayleigh, 1.22 * lam / D)
    fig, axes = plt.subplots(1, 4, figsize=(15.2, 3.5), constrained_layout=True)
    axes[0].plot(xa * 1e6, A)
    span_um = min(np.ptp(xa), 140e-6) * 1e6
    axes[0].set_xlim(-span_um / 2, span_um / 2)
    axes[0].set_xlabel("aperture x (um)")
    axes[0].set_ylabel("transmission")
    axes[0].set_title(kind)
    axes[1].plot(theta_deg, I)
    axes[1].set_xlim(-20, 20)
    axes[1].set_xlabel("angle theta (deg)")
    axes[1].set_ylabel("normalized intensity")
    axes[1].set_title(f"Fraunhofer FFT, lambda={wavelength_nm:.0f} nm")
    if kind in ("double slit", "grating"):
        for m in range(-3, 4):
            s = m * lam / separation
            if abs(s) < np.sin(np.radians(20)):
                axes[1].axvline(np.degrees(np.arcsin(s)), color="k", alpha=0.18)
    theta_sweep = np.linspace(-20, 20, 1000)
    for lam_nm, color in [(430, "tab:blue"), (550, "tab:green"), (680, "tab:red")]:
        inten = double_slit_fraunhofer(theta_sweep, lam_nm, slit_width_um, separation_um)
        axes[2].plot(theta_sweep, inten, color=color, label=f"{lam_nm} nm")
    axes[2].set_xlim(-20, 20)
    axes[2].set_xlabel("angle theta (deg)")
    axes[2].set_ylabel("normalized intensity")
    axes[2].set_title("Double slit: wavelength shifts peaks")
    axes[2].legend(fontsize=8)
    axes[3].plot(np.degrees(theta_airy), airy)
    axes[3].axvline(np.degrees(rayleigh), color="tab:red", ls="--", label=f"Rayleigh {np.degrees(rayleigh):.2f} deg")
    axes[3].set_xlim(0, min(30, np.degrees(theta_airy.max())))
    axes[3].set_xlabel("angle theta (deg)")
    axes[3].set_ylabel("Airy intensity")
    axes[3].set_title(f"Abbe d_min={abbe*1e9:.0f} nm")
    axes[3].legend(fontsize=8)
    plt.show()


def double_slit_wavelength_animation(slit_width_um=8.0, separation_um=24.0, lambda_min_nm=400.0, lambda_max_nm=750.0, n_frames=50):
    theta = np.linspace(-22, 22, 1000)
    wavelengths = np.linspace(lambda_min_nm, lambda_max_nm, n_frames)
    frames = [double_slit_fraunhofer(theta, lam_nm, slit_width_um, separation_um) for lam_nm in wavelengths]
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    line, = ax.plot(theta, frames[0], lw=2, color="tab:blue")
    m_lines = [ax.axvline(0, color="0.2", alpha=0.15, lw=1) for _ in range(6)]
    ax.set_xlim(theta.min(), theta.max())
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("angle theta (deg)")
    ax.set_ylabel("normalized intensity")
    ax.set_title("Double-slit Fraunhofer diffraction")

    def update(i):
        lam_nm = wavelengths[i]
        line.set_ydata(frames[i])
        for line_obj, m in zip(m_lines, [-3, -2, -1, 1, 2, 3]):
            s = m * lam_nm * 1e-9 / (separation_um * 1e-6)
            if abs(s) < np.sin(np.radians(theta.max())):
                line_obj.set_xdata([np.degrees(np.arcsin(s)), np.degrees(np.arcsin(s))])
                line_obj.set_alpha(0.18)
            else:
                line_obj.set_alpha(0.0)
        ax.set_title(f"Double-slit Fraunhofer diffraction, lambda={lam_nm:.0f} nm")
        return (line, *m_lines)

    anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=80, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


interact(plot_diffraction,
         kind=Dropdown(options=["single slit", "double slit", "grating"], value="double slit", description="aperture"),
         wavelength_nm=FloatSlider(value=550, min=400, max=750, step=10, description="lambda"),
         slit_width_um=FloatSlider(value=8, min=2, max=30, step=1, description="width"),
         separation_um=FloatSlider(value=24, min=6, max=80, step=2, description="d"),
         grating_lines=IntSlider(value=5, min=2, max=12, step=1, description="lines"),
         NA=FloatSlider(value=1.0, min=0.2, max=1.4, step=0.1, description="NA"))

anim_width = FloatSlider(value=8, min=2, max=30, step=1, description="width")
anim_separation = FloatSlider(value=24, min=6, max=80, step=2, description="d")
anim_lambda_min = FloatSlider(value=400, min=380, max=650, step=10, description="lambda min")
anim_lambda_max = FloatSlider(value=750, min=500, max=780, step=10, description="lambda max")
start_diffraction_button = Button(description="Start wavelength animation", button_style="primary")
diffraction_animation_output = Output()


def start_diffraction_animation(_):
    diffraction_animation_output.clear_output(wait=True)
    with diffraction_animation_output:
        display(double_slit_wavelength_animation(
            slit_width_um=anim_width.value,
            separation_um=anim_separation.value,
            lambda_min_nm=anim_lambda_min.value,
            lambda_max_nm=max(anim_lambda_max.value, anim_lambda_min.value + 10),
        ))


start_diffraction_button.on_click(start_diffraction_animation)
display(anim_width, anim_separation, anim_lambda_min, anim_lambda_max, start_diffraction_button, diffraction_animation_output)
note('Press Start to see wavelength push double-slit fringes to larger angles.')
